In [ ]:
gamma = 0.9
theta = 1e-10

states = ["S0", "S1", "G"]
non_terminal_states = ["S0", "S1"]
actions = ["Up", "Down", "Left", "Right"]

# Grid World
# S0   S1
# WALL G

grid = [
    ["S0", "S1"],
    ["WALL", "G"]
]

positions = {
    "S0": (0, 0),
    "S1": (0, 1),
    "G": (1, 1)
}

wall = (1, 0)

deltas = {
    "Up": (-1, 0),
    "Down": (1, 0),
    "Left": (0, -1),
    "Right": (0, 1)
}

# Define transitions and rewards
transition = {}
reward = {}

for s in non_terminal_states:
    r, c = positions[s]

    for a in actions:
        dr, dc = deltas[a]
        nr, nc = r + dr, c + dc

        if not (0 <= nr < 2 and 0 <= nc < 2) or (nr, nc) == wall:
            next_state = s
            immediate_reward = -1
        else:
            next_state = grid[nr][nc]
            immediate_reward = 100 if next_state == "G" else -1

        transition[(s, a)] = next_state
        reward[(s, a)] = immediate_reward

# Display MDP information
print("STATES:", states)
print("ACTIONS:", actions)
print("Discount factor:", gamma)
print("Normal reward: -1")
print("Wall/boundary reward: -1")
print("Goal reward: +100")
print("Terminal state value: V(G) = 0")

print("\nTRANSITION MAPPING")
for s in non_terminal_states:
    for a in actions:
        print(
            f"State={s}, Action={a}, "
            f"Next State={transition[(s, a)]}, "
            f"Reward={reward[(s, a)]}"
        )

# Initialize policy and state values
policy = {
    "S0": "Up",
    "S1": "Up"
}

V = {s: 0.0 for s in states}

print("\nINITIAL POLICY:", policy)
print("INITIAL VALUES:", V)

# Policy Iteration
policy_iteration = 0

while True:
    policy_iteration += 1

    print(f"\n===== POLICY ITERATION {policy_iteration} =====")
    print("Current Policy:", policy)

    # 1. Policy Evaluation
    evaluation_iteration = 0

    while True:
        new_V = V.copy()

        for s in non_terminal_states:
            a = policy[s]
            next_state = transition[(s, a)]

            new_V[s] = (
                reward[(s, a)] + gamma * V[next_state]
            )

        new_V["G"] = 0.0

        delta = max(abs(new_V[s] - V[s]) for s in states)
        V = new_V
        evaluation_iteration += 1

        print(
            f"Evaluation {evaluation_iteration}: "
            f"V(S0)={V['S0']:.6f}, "
            f"V(S1)={V['S1']:.6f}, "
            f"V(G)={V['G']:.6f}, "
            f"Delta={delta:.6f}"
        )

        if delta < theta:
            break

    # 2. Calculate Q values
    Q = {}

    print("\nACTION VALUES Q_pi(s, a)")

    for s in non_terminal_states:
        for a in actions:
            next_state = transition[(s, a)]

            Q[(s, a)] = (
                reward[(s, a)] + gamma * V[next_state]
            )

            print(f"Q({s}, {a}) = {Q[(s, a)]:.6f}")

    # 3. Policy Improvement
    improved_policy = {}

    for s in non_terminal_states:
        max_q = max(Q[(s, a)] for a in actions)

        best_actions = [
            a for a in actions
            if abs(Q[(s, a)] - max_q) < 1e-9
        ]

        # Keep the current action if it is tied for best.
        if policy[s] in best_actions:
            improved_policy[s] = policy[s]
        else:
            improved_policy[s] = best_actions[0]

    print("\nImproved Policy:", improved_policy)

    # 4. Check policy stability
    if improved_policy == policy:
        print("Policy is stable.")
        break

    policy = improved_policy

# Final state values
print("\n===== FINAL OPTIMAL STATE VALUES =====")

for s in states:
    print(f"V*({s}) = {V[s]:.6f}")

# Final optimal policy
print("\n===== OPTIMAL POLICY =====")

for s in non_terminal_states:
    print(f"Policy({s}) = {policy[s]}")

# Final action-value table
print("\n===== FINAL ACTION-VALUE TABLE =====")
print(f"{'State':<8}{'Up':>12}{'Down':>12}{'Left':>12}{'Right':>12}")

for s in non_terminal_states:
    print(
        f"{s:<8}"
        f"{Q[(s, 'Up')]:>12.4f}"
        f"{Q[(s, 'Down')]:>12.4f}"
        f"{Q[(s, 'Left')]:>12.4f}"
        f"{Q[(s, 'Right')]:>12.4f}"
    )

# Bellman Optimality verification
print("\n===== BELLMAN OPTIMALITY VERIFICATION =====")

for s in non_terminal_states:
    max_q = max(Q[(s, a)] for a in actions)
    difference = abs(V[s] - max_q)

    print(
        f"State={s}, V*={V[s]:.6f}, "
        f"max Q*={max_q:.6f}, "
        f"Difference={difference:.10f}"
    )

    assert difference < 1e-8

print("\nBellman Optimality verification passed.")